In [1]:
# =============================================================================
# 13_compare_image_quality_across_datasets.ipynb
#
# PURPOSE
# =============================================================================
#
# Create a SIMPLE quality comparison for:
#
#   1. Daily / acquisition-level data
#   2. Weekly / 7-day composites
#   3. Biweekly / 14-day composites
#
#
# MAIN QUESTION
# =============================================================================
#
# For each treatment site and its 5 matched counterfactual sites:
#
#   - How many images / periods are available?
#   - How many have >= 80% valid pixels?
#   - What percentage are >= 80%?
#
#
# For WEEKLY and BIWEEKLY:
#
#   denominator = ALL scheduled periods
#
# Therefore:
#
#   usable_period_rate =
#
#       number of periods with >=80% valid pixels
#       ------------------------------------------
#       number of expected periods
#
#
# A missing period is NOT interpreted as a 0%-quality image.
#
# However, it IS counted as a period that is not usable because there is
# no image available for that scheduled period.
#
#
# For DAILY:
#
# There is no fixed daily panel because Sentinel imagery is not acquired
# every calendar day.
#
# Therefore:
#
#   denominator = actual acquisition images
#
#
# OUTPUT
# =============================================================================
#
# finals/
# └── quality/
#
#     ├── image_quality_simple.xlsx
#     └── image_quality_simple.csv
#
#
# EXCEL SHEETS
# =============================================================================
#
#   README
#
#   Daily_S1
#   Daily_S2
#
#   Weekly_S1
#   Weekly_S2
#
#   Biweekly_S1
#   Biweekly_S2
#
#
# Each sheet contains one row for:
#
#   treatment × before/after
#
# and shows:
#
#   - treatment quality
#   - quality of its 5 counterfactual units
#   - pooled quality of the 5 counterfactual units
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings(
    "ignore"
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


WEEKLY_DIR = (
    FINALS_DIR /
    "weekly_datasets"
)


BIWEEKLY_DIR = (
    FINALS_DIR /
    "biweekly_datasets"
)


QUALITY_DIR = (
    FINALS_DIR /
    "quality"
)


QUALITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. Input files
# =============================================================================

SELECTED_SAMPLE_FILE = (
    DAILY_DIR /
    "selected_site_sample.csv"
)


DAILY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


WEEKLY_FILE = (
    WEEKLY_DIR /
    "weekly_image_quality.csv"
)


BIWEEKLY_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.csv"
)


# =============================================================================
# 4. Outputs
# =============================================================================

OUTPUT_EXCEL = (
    QUALITY_DIR /
    "image_quality_simple.xlsx"
)


OUTPUT_CSV = (
    QUALITY_DIR /
    "image_quality_simple.csv"
)


# =============================================================================
# 5. Study design
# =============================================================================

HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


STUDY_START = pd.Timestamp(
    "2024-05-10"
)


STUDY_END = pd.Timestamp(
    "2025-02-13"
)


QUALITY_THRESHOLD = 0.80


EXPECTED_TREATMENT_SITES = 10


EXPECTED_CONTROLS_PER_TREATMENT = 5


# =============================================================================
# 6. Check files
# =============================================================================

required_files = [

    SELECTED_SAMPLE_FILE,

    DAILY_FILE,

    WEEKLY_FILE,

    BIWEEKLY_FILE,

]


missing_files = [

    file_path

    for file_path in required_files

    if not file_path.exists()

]


if missing_files:

    raise FileNotFoundError(
        "Required files are missing:\n\n"
        +
        "\n".join(
            str(file_path)
            for file_path in missing_files
        )
    )


print(
    "\nAll required source files found."
)


# =============================================================================
# 7. Helper functions
# =============================================================================

def normalize_group(value):

    value = str(
        value
    ).strip().lower()


    if value in [
        "treatment",
        "treated",
    ]:

        return "treatment"


    if value in [
        "counterfactual",
        "control",
        "controls",
    ]:

        return "counterfactual"


    return value


def normalize_sensor(value):

    value = str(
        value
    ).strip().lower()


    if value in [
        "sentinel1",
        "sentinel-1",
        "s1",
    ]:

        return "sentinel1"


    if value in [
        "sentinel2",
        "sentinel-2",
        "s2",
    ]:

        return "sentinel2"


    return value


def identify_quality_column(dataframe):

    candidates = [

        "valid_pixel_fraction",

        "valid_pixel_fraction_any_band",

        "valid_pixel_fraction_all_bands",

    ]


    for column in candidates:

        if column in dataframe.columns:

            return column


    raise ValueError(
        "No valid-pixel quality column found."
    )


def assign_before_after(date):

    date = pd.Timestamp(
        date
    )


    if date < HELENE_REFERENCE_DATE:

        return "before"


    return "after"


# =============================================================================
# 8. Load selected sample
# =============================================================================

sample = pd.read_csv(
    SELECTED_SAMPLE_FILE
)


required_sample_columns = [

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

]


missing_sample_columns = [

    column

    for column in required_sample_columns

    if column not in sample.columns

]


if missing_sample_columns:

    raise ValueError(
        "selected_site_sample.csv is missing columns:\n"
        +
        str(
            missing_sample_columns
        )
    )


sample[
    "site_id"
] = (
    sample[
        "site_id"
    ]
    .astype(str)
)


sample[
    "group"
] = (
    sample[
        "group"
    ]
    .apply(
        normalize_group
    )
)


sample[
    "matched_treatment_site_id"
] = (
    sample[
        "matched_treatment_site_id"
    ]
    .astype("string")
)


sample[
    "control_rank"
] = pd.to_numeric(
    sample[
        "control_rank"
    ],
    errors="coerce",
)


# Treatment sites map to themselves.

treatment_mask = (
    sample[
        "group"
    ]
    ==
    "treatment"
)


sample.loc[
    treatment_mask,
    "matched_treatment_site_id",
] = (
    sample.loc[
        treatment_mask,
        "site_id",
    ]
)


sample = (
    sample
    .drop_duplicates(
        subset=[
            "site_id",
            "group",
        ]
    )
    .copy()
)


# =============================================================================
# 9. Validate matching
# =============================================================================

treatment_sites = (
    sample
    .loc[
        sample[
            "group"
        ]
        ==
        "treatment",
        "site_id",
    ]
    .drop_duplicates()
    .tolist()
)


control_sample = (
    sample
    .loc[
        sample[
            "group"
        ]
        ==
        "counterfactual"
    ]
    .copy()
)


controls_per_treatment = (
    control_sample
    .groupby(
        "matched_treatment_site_id"
    )[
        "site_id"
    ]
    .nunique()
)


print(
    "\nTreatment sites:"
)


print(
    len(
        treatment_sites
    )
)


print(
    "\nControls per treatment:"
)


print(
    controls_per_treatment
)


if len(
    treatment_sites
) != EXPECTED_TREATMENT_SITES:

    print(
        "\nWARNING:"
    )


    print(
        "Expected 10 treatment sites but found",
        len(
            treatment_sites
        )
    )


if not (
    controls_per_treatment
    ==
    EXPECTED_CONTROLS_PER_TREATMENT
).all():

    print(
        "\nWARNING:"
    )


    print(
        "At least one treatment does not have exactly 5 controls."
    )


# =============================================================================
# 10. Metadata used for all datasets
# =============================================================================

sample_metadata = (
    sample[
        [
            "site_id",
            "group",
            "matched_treatment_site_id",
            "control_rank",
        ]
    ]
    .drop_duplicates()
)


selected_site_ids = set(
    sample[
        "site_id"
    ]
)


# =============================================================================
# 11. Standardize one dataset
# =============================================================================

def standardize_dataset(
    dataframe,
    dataset_name,
):

    df = (
        dataframe.copy()
    )


    # -------------------------------------------------------------------------
    # IDs
    # -------------------------------------------------------------------------

    df[
        "site_id"
    ] = (
        df[
            "site_id"
        ]
        .astype(str)
    )


    df[
        "group"
    ] = (
        df[
            "group"
        ]
        .apply(
            normalize_group
        )
    )


    df[
        "sensor"
    ] = (
        df[
            "sensor"
        ]
        .apply(
            normalize_sensor
        )
    )


    # -------------------------------------------------------------------------
    # Selected sample only
    # -------------------------------------------------------------------------

    df = (
        df
        .loc[
            df[
                "site_id"
            ]
            .isin(
                selected_site_ids
            )
        ]
        .copy()
    )


    # -------------------------------------------------------------------------
    # Quality
    # -------------------------------------------------------------------------

    quality_column = (
        identify_quality_column(
            df
        )
    )


    df[
        "quality_fraction"
    ] = pd.to_numeric(
        df[
            quality_column
        ],
        errors="coerce",
    )


    # -------------------------------------------------------------------------
    # Remove old matching columns before merge
    # -------------------------------------------------------------------------

    for column in [

        "matched_treatment_site_id",

        "control_rank",

    ]:

        if column in df.columns:

            df = (
                df.drop(
                    columns=column
                )
            )


    df = (
        df.merge(
            sample_metadata,
            on=[
                "site_id",
                "group",
            ],
            how="left",
        )
    )


    df[
        "dataset"
    ] = (
        dataset_name
    )


    return df


# =============================================================================
# 12. Load DAILY
# =============================================================================

daily = pd.read_csv(
    DAILY_FILE
)


# Keep successful / existing daily TIFFs when status exists.

if "status" in daily.columns:

    daily = (
        daily
        .loc[
            daily[
                "status"
            ]
            .isin(
                [
                    "success",
                    "existing",
                ]
            )
        ]
        .copy()
    )


daily = (
    standardize_dataset(
        daily,
        "daily",
    )
)


daily[
    "acquisition_date"
] = pd.to_datetime(
    daily[
        "acquisition_date"
    ],
    errors="coerce",
)


daily = (
    daily
    .loc[
        daily[
            "acquisition_date"
        ]
        .between(
            STUDY_START,
            STUDY_END,
            inclusive="both",
        )
    ]
    .copy()
)


daily[
    "period"
] = (
    daily[
        "acquisition_date"
    ]
    .apply(
        assign_before_after
    )
)


daily[
    "time_id"
] = (
    daily[
        "acquisition_date"
    ]
    .dt.strftime(
        "%Y-%m-%d"
    )
)


# Daily rows represent acquisitions.

daily[
    "image_available"
] = (
    daily[
        "quality_fraction"
    ]
    .notna()
    .astype(int)
)


# =============================================================================
# 13. Load WEEKLY
# =============================================================================

weekly = pd.read_csv(
    WEEKLY_FILE
)


weekly = (
    standardize_dataset(
        weekly,
        "weekly",
    )
)


if "week_id" in weekly.columns:

    weekly[
        "time_id"
    ] = (
        weekly[
            "week_id"
        ]
        .astype(str)
    )


else:

    weekly[
        "time_id"
    ] = (
        weekly[
            "period"
        ].astype(str)
        +
        "_W"
        +
        weekly[
            "week_number"
        ]
        .astype(int)
        .astype(str)
        .str.zfill(
            2
        )
    )


if "has_any_acquisition" in weekly.columns:

    weekly[
        "image_available"
    ] = pd.to_numeric(
        weekly[
            "has_any_acquisition"
        ],
        errors="coerce",
    ).fillna(
        0
    ).astype(int)


elif "has_data" in weekly.columns:

    weekly[
        "image_available"
    ] = pd.to_numeric(
        weekly[
            "has_data"
        ],
        errors="coerce",
    ).fillna(
        0
    ).astype(int)


else:

    weekly[
        "image_available"
    ] = (
        weekly[
            "quality_fraction"
        ]
        .notna()
        .astype(int)
    )


# =============================================================================
# 14. Load BIWEEKLY
# =============================================================================

biweekly = pd.read_csv(
    BIWEEKLY_FILE
)


biweekly = (
    standardize_dataset(
        biweekly,
        "biweekly",
    )
)


if "period_id" in biweekly.columns:

    biweekly[
        "time_id"
    ] = (
        biweekly[
            "period_id"
        ]
        .astype(str)
    )


else:

    biweekly[
        "time_id"
    ] = (
        biweekly[
            "period"
        ].astype(str)
        +
        "_P"
        +
        biweekly[
            "period_number"
        ]
        .astype(int)
        .astype(str)
        .str.zfill(
            2
        )
    )


if "has_data" in biweekly.columns:

    biweekly[
        "image_available"
    ] = pd.to_numeric(
        biweekly[
            "has_data"
        ],
        errors="coerce",
    ).fillna(
        0
    ).astype(int)


else:

    biweekly[
        "image_available"
    ] = (
        biweekly[
            "quality_fraction"
        ]
        .notna()
        .astype(int)
    )


# =============================================================================
# 15. Combine
# =============================================================================

columns_needed = [

    "dataset",

    "sensor",

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

    "period",

    "time_id",

    "image_available",

    "quality_fraction",

]


quality_data = pd.concat(

    [

        daily[
            columns_needed
        ],

        weekly[
            columns_needed
        ],

        biweekly[
            columns_needed
        ],

    ],

    ignore_index=True,

)


# =============================================================================
# 16. Define >=80% usable image
# =============================================================================

quality_data[
    "quality_ge_80"
] = (

    quality_data[
        "quality_fraction"
    ]
    >=
    QUALITY_THRESHOLD

).astype(int)


# =============================================================================
# 17. SITE-LEVEL QUALITY SUMMARY
#
# This is the core calculation.
# =============================================================================

site_summary_records = []


group_columns = [

    "dataset",

    "sensor",

    "period",

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

]


for keys, subset in (
    quality_data
    .groupby(
        group_columns,
        dropna=False,
    )
):

    (

        dataset_name,

        sensor,

        period,

        site_id,

        group,

        matched_treatment_site_id,

        control_rank,

    ) = keys


    # -------------------------------------------------------------------------
    # Daily:
    #
    # denominator = actual acquisition images
    # -------------------------------------------------------------------------

    if dataset_name == "daily":

        expected_periods = (
            subset[
                "time_id"
            ]
            .nunique()
        )


    # -------------------------------------------------------------------------
    # Weekly / biweekly:
    #
    # denominator = scheduled periods represented in panel
    # -------------------------------------------------------------------------

    else:

        expected_periods = (
            subset[
                "time_id"
            ]
            .nunique()
        )


    available_periods = int(
        subset[
            "image_available"
        ]
        .sum()
    )


    periods_ge_80 = int(
        subset[
            "quality_ge_80"
        ]
        .sum()
    )


    missing_periods = (
        expected_periods
        -
        available_periods
    )


    if expected_periods > 0:

        percent_expected_ge_80 = (
            periods_ge_80
            /
            expected_periods
            *
            100
        )


        availability_percentage = (
            available_periods
            /
            expected_periods
            *
            100
        )


    else:

        percent_expected_ge_80 = (
            np.nan
        )


        availability_percentage = (
            np.nan
        )


    if available_periods > 0:

        percent_available_ge_80 = (
            periods_ge_80
            /
            available_periods
            *
            100
        )


    else:

        percent_available_ge_80 = (
            np.nan
        )


    measurable_quality = (
        subset[
            "quality_fraction"
        ]
        .dropna()
    )


    if len(
        measurable_quality
    ) > 0:

        mean_quality_percentage = (
            measurable_quality.mean()
            *
            100
        )


        median_quality_percentage = (
            measurable_quality.median()
            *
            100
        )


    else:

        mean_quality_percentage = (
            np.nan
        )


        median_quality_percentage = (
            np.nan
        )


    site_summary_records.append(
        {

            "dataset":
                dataset_name,

            "sensor":
                sensor,

            "period":
                period,

            "site_id":
                site_id,

            "group":
                group,

            "matched_treatment_site_id":
                matched_treatment_site_id,

            "control_rank":
                control_rank,

            "expected_periods_or_images":
                expected_periods,

            "available_periods_or_images":
                available_periods,

            "missing_periods":
                missing_periods,

            "periods_ge_80pct":
                periods_ge_80,

            "percent_expected_ge_80pct":
                percent_expected_ge_80,

            "percent_available_ge_80pct":
                percent_available_ge_80,

            "availability_percentage":
                availability_percentage,

            "mean_quality_percentage":
                mean_quality_percentage,

            "median_quality_percentage":
                median_quality_percentage,

        }
    )


site_summary = pd.DataFrame(
    site_summary_records
)


# =============================================================================
# 18. Treatment rows
# =============================================================================

treatment_quality = (
    site_summary
    .loc[
        site_summary[
            "group"
        ]
        ==
        "treatment"
    ]
    .copy()
)


# =============================================================================
# 19. Counterfactual rows
# =============================================================================

control_quality = (
    site_summary
    .loc[
        site_summary[
            "group"
        ]
        ==
        "counterfactual"
    ]
    .copy()
)


# =============================================================================
# 20. Pooled counterfactual summary
#
# For each treatment site:
#
# combine its 5 controls.
# =============================================================================

control_pool = (
    control_quality
    .groupby(
        [
            "dataset",
            "sensor",
            "period",
            "matched_treatment_site_id",
        ],
        as_index=False,
    )
    .agg(

        number_of_controls=(
            "site_id",
            "nunique",
        ),

        control_expected_periods=(
            "expected_periods_or_images",
            "sum",
        ),

        control_available_periods=(
            "available_periods_or_images",
            "sum",
        ),

        control_missing_periods=(
            "missing_periods",
            "sum",
        ),

        control_periods_ge_80pct=(
            "periods_ge_80pct",
            "sum",
        ),

        control_mean_quality_percentage=(
            "mean_quality_percentage",
            "mean",
        ),

    )
)


control_pool[
    "control_percent_expected_ge_80pct"
] = np.where(

    control_pool[
        "control_expected_periods"
    ]
    >
    0,

    control_pool[
        "control_periods_ge_80pct"
    ]
    /
    control_pool[
        "control_expected_periods"
    ]
    *
    100,

    np.nan,

)


control_pool[
    "control_percent_available_ge_80pct"
] = np.where(

    control_pool[
        "control_available_periods"
    ]
    >
    0,

    control_pool[
        "control_periods_ge_80pct"
    ]
    /
    control_pool[
        "control_available_periods"
    ]
    *
    100,

    np.nan,

)


control_pool[
    "control_availability_percentage"
] = np.where(

    control_pool[
        "control_expected_periods"
    ]
    >
    0,

    control_pool[
        "control_available_periods"
    ]
    /
    control_pool[
        "control_expected_periods"
    ]
    *
    100,

    np.nan,

)


# =============================================================================
# 21. Rename treatment columns
# =============================================================================

treatment_quality = (
    treatment_quality
    .rename(
        columns={

            "site_id":
                "treatment_site",

            "expected_periods_or_images":
                "treatment_expected_periods",

            "available_periods_or_images":
                "treatment_available_periods",

            "missing_periods":
                "treatment_missing_periods",

            "periods_ge_80pct":
                "treatment_periods_ge_80pct",

            "percent_expected_ge_80pct":
                "treatment_percent_expected_ge_80pct",

            "percent_available_ge_80pct":
                "treatment_percent_available_ge_80pct",

            "availability_percentage":
                "treatment_availability_percentage",

            "mean_quality_percentage":
                "treatment_mean_quality_percentage",

            "median_quality_percentage":
                "treatment_median_quality_percentage",

        }
    )
)


# =============================================================================
# 22. Merge treatment with pooled controls
# =============================================================================

simple_summary = (
    treatment_quality
    .merge(

        control_pool,

        left_on=[
            "dataset",
            "sensor",
            "period",
            "treatment_site",
        ],

        right_on=[
            "dataset",
            "sensor",
            "period",
            "matched_treatment_site_id",
        ],

        how="left",

        suffixes=(
            "",
            "_control",
        ),

    )
)


# =============================================================================
# 23. Keep SIMPLE columns
# =============================================================================

simple_columns = [

    "dataset",

    "sensor",

    "period",

    "treatment_site",

    # Treatment
    "treatment_expected_periods",

    "treatment_available_periods",

    "treatment_missing_periods",

    "treatment_periods_ge_80pct",

    "treatment_percent_expected_ge_80pct",

    "treatment_percent_available_ge_80pct",

    "treatment_mean_quality_percentage",

    # Controls
    "number_of_controls",

    "control_expected_periods",

    "control_available_periods",

    "control_missing_periods",

    "control_periods_ge_80pct",

    "control_percent_expected_ge_80pct",

    "control_percent_available_ge_80pct",

    "control_mean_quality_percentage",

]


simple_summary = (
    simple_summary[
        simple_columns
    ]
    .copy()
)


# =============================================================================
# 24. Add individual control information
#
# Control 1 ... Control 5
# =============================================================================

control_wide_records = []


for keys, subset in (
    control_quality
    .groupby(
        [
            "dataset",
            "sensor",
            "period",
            "matched_treatment_site_id",
        ]
    )
):

    (

        dataset_name,

        sensor,

        period,

        treatment_site,

    ) = keys


    subset = (
        subset
        .sort_values(
            [
                "control_rank",
                "site_id",
            ]
        )
        .copy()
    )


    record = {

        "dataset":
            dataset_name,

        "sensor":
            sensor,

        "period":
            period,

        "treatment_site":
            treatment_site,

    }


    for _, row in subset.iterrows():

        if pd.notna(
            row[
                "control_rank"
            ]
        ):

            rank = int(
                row[
                    "control_rank"
                ]
            )


        else:

            continue


        record[
            f"control_{rank}_site"
        ] = (
            row[
                "site_id"
            ]
        )


        record[
            f"control_{rank}_expected"
        ] = (
            row[
                "expected_periods_or_images"
            ]
        )


        record[
            f"control_{rank}_available"
        ] = (
            row[
                "available_periods_or_images"
            ]
        )


        record[
            f"control_{rank}_ge80"
        ] = (
            row[
                "periods_ge_80pct"
            ]
        )


        record[
            f"control_{rank}_percent_ge80"
        ] = (
            row[
                "percent_expected_ge_80pct"
            ]
        )


        record[
            f"control_{rank}_mean_quality"
        ] = (
            row[
                "mean_quality_percentage"
            ]
        )


    control_wide_records.append(
        record
    )


control_wide = pd.DataFrame(
    control_wide_records
)


# =============================================================================
# 25. Final Excel table
# =============================================================================

final_summary = (
    simple_summary
    .merge(

        control_wide,

        on=[
            "dataset",
            "sensor",
            "period",
            "treatment_site",
        ],

        how="left",

    )
)


# =============================================================================
# 26. Order columns
# =============================================================================

base_columns = [

    "dataset",

    "sensor",

    "period",

    "treatment_site",

    "treatment_expected_periods",

    "treatment_available_periods",

    "treatment_periods_ge_80pct",

    "treatment_percent_expected_ge_80pct",

    "treatment_mean_quality_percentage",

]


control_columns = []


for rank in range(
    1,
    EXPECTED_CONTROLS_PER_TREATMENT + 1,
):

    control_columns.extend(
        [

            f"control_{rank}_site",

            f"control_{rank}_expected",

            f"control_{rank}_available",

            f"control_{rank}_ge80",

            f"control_{rank}_percent_ge80",

            f"control_{rank}_mean_quality",

        ]
    )


pooled_columns = [

    "number_of_controls",

    "control_expected_periods",

    "control_available_periods",

    "control_periods_ge_80pct",

    "control_percent_expected_ge_80pct",

    "control_mean_quality_percentage",

]


final_columns = (

    base_columns
    +
    control_columns
    +
    pooled_columns

)


for column in final_columns:

    if column not in final_summary.columns:

        final_summary[
            column
        ] = np.nan


final_summary = (
    final_summary[
        final_columns
    ]
    .copy()
)


# =============================================================================
# 27. Sort
# =============================================================================

dataset_order = {

    "daily":
        1,

    "weekly":
        2,

    "biweekly":
        3,

}


sensor_order = {

    "sentinel1":
        1,

    "sentinel2":
        2,

}


period_order = {

    "before":
        1,

    "after":
        2,

}


final_summary[
    "_dataset_order"
] = (
    final_summary[
        "dataset"
    ]
    .map(
        dataset_order
    )
)


final_summary[
    "_sensor_order"
] = (
    final_summary[
        "sensor"
    ]
    .map(
        sensor_order
    )
)


final_summary[
    "_period_order"
] = (
    final_summary[
        "period"
    ]
    .map(
        period_order
    )
)


final_summary = (
    final_summary
    .sort_values(
        [
            "_dataset_order",
            "_sensor_order",
            "treatment_site",
            "_period_order",
        ]
    )
    .drop(
        columns=[
            "_dataset_order",
            "_sensor_order",
            "_period_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 28. Round percentages
# =============================================================================

percentage_columns = [

    column

    for column in final_summary.columns

    if (
        "percent" in column
        or
        "mean_quality" in column
    )

]


for column in percentage_columns:

    final_summary[
        column
    ] = (
        pd.to_numeric(
            final_summary[
                column
            ],
            errors="coerce",
        )
        .round(
            1
        )
    )


# =============================================================================
# 29. Save simple CSV
# =============================================================================

final_summary.to_csv(
    OUTPUT_CSV,
    index=False,
)


# =============================================================================
# 30. README
# =============================================================================

readme = pd.DataFrame(
    {

        "Item": [

            "Purpose",

            "Quality threshold",

            "Quality definition",

            "Daily denominator",

            "Weekly denominator",

            "Biweekly denominator",

            "Missing period",

            "Treatment",

            "Counterfactual",

            "Pooled controls",

        ],

        "Definition": [

            (
                "Compare image usability for each treatment site "
                "and its five matched counterfactual sites."
            ),

            "At least 80% valid spatial pixels.",

            (
                "A spatial pixel is valid when at least one output "
                "band has a finite value."
            ),

            (
                "Actual satellite acquisitions. Daily data do not "
                "represent one image for every calendar day."
            ),

            (
                "All scheduled weekly periods represented in the "
                "Notebook 11 fixed panel."
            ),

            (
                "All scheduled 14-day periods represented in the "
                "Notebook 12 fixed panel."
            ),

            (
                "Missing weekly/biweekly imagery is not assigned "
                "0% image quality, but the period does not count "
                "as >=80% usable."
            ),

            (
                "Quality statistics for the treatment spatial unit."
            ),

            (
                "Each control_1 ... control_5 corresponds to one "
                "matched counterfactual spatial unit."
            ),

            (
                "Control totals combine all five matched "
                "counterfactual units."
            ),

        ],

    }
)


# =============================================================================
# 31. Excel formatting helper
# =============================================================================

def format_sheet(
    worksheet,
):

    worksheet.freeze_panes = (
        "A2"
    )


    worksheet.auto_filter.ref = (
        worksheet.dimensions
    )


    # Header
    for cell in worksheet[
        1
    ]:

        cell.font = cell.font.copy(
            bold=True
        )


    # Reasonable widths
    for column_cells in worksheet.columns:

        column_letter = (
            column_cells[
                0
            ].column_letter
        )


        max_length = 0


        for cell in column_cells:

            if cell.value is not None:

                max_length = max(
                    max_length,
                    len(
                        str(
                            cell.value
                        )
                    ),
                )


        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(
                max_length + 2,
                10,
            ),
            28,
        )


# =============================================================================
# 32. Write SIMPLE Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        OUTPUT_EXCEL,
        engine="openpyxl",
    ) as writer:

        # ---------------------------------------------------------------------
        # README
        # ---------------------------------------------------------------------

        readme.to_excel(
            writer,
            sheet_name="README",
            index=False,
        )


        # ---------------------------------------------------------------------
        # One very simple sheet per dataset × sensor
        # ---------------------------------------------------------------------

        sheet_definitions = [

            (
                "daily",
                "sentinel1",
                "Daily_S1",
            ),

            (
                "daily",
                "sentinel2",
                "Daily_S2",
            ),

            (
                "weekly",
                "sentinel1",
                "Weekly_S1",
            ),

            (
                "weekly",
                "sentinel2",
                "Weekly_S2",
            ),

            (
                "biweekly",
                "sentinel1",
                "Biweekly_S1",
            ),

            (
                "biweekly",
                "sentinel2",
                "Biweekly_S2",
            ),

        ]


        for (
            dataset_name,
            sensor_name,
            sheet_name,
        ) in sheet_definitions:

            sheet_data = (
                final_summary
                .loc[
                    (
                        final_summary[
                            "dataset"
                        ]
                        ==
                        dataset_name
                    )
                    &
                    (
                        final_summary[
                            "sensor"
                        ]
                        ==
                        sensor_name
                    )
                ]
                .copy()
            )


            # Dataset/sensor are already encoded in sheet name.
            sheet_data = (
                sheet_data.drop(
                    columns=[
                        "dataset",
                        "sensor",
                    ]
                )
            )


            sheet_data.to_excel(
                writer,
                sheet_name=
                    sheet_name,
                index=False,
            )


        # ---------------------------------------------------------------------
        # Formatting
        # ---------------------------------------------------------------------

        workbook = (
            writer.book
        )


        for worksheet in workbook.worksheets:

            format_sheet(
                worksheet
            )


    print(
        "\nSimple Excel workbook saved:"
    )


    print(
        OUTPUT_EXCEL
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


    print(
        "Install with:"
    )


    print(
        "%pip install openpyxl"
    )


# =============================================================================
# 33. Print simple comparison
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "SIMPLE IMAGE QUALITY COMPARISON"
)


print(
    "=" * 120
)


print_columns = [

    "dataset",

    "sensor",

    "period",

    "treatment_site",

    "treatment_expected_periods",

    "treatment_available_periods",

    "treatment_periods_ge_80pct",

    "treatment_percent_expected_ge_80pct",

    "number_of_controls",

    "control_expected_periods",

    "control_available_periods",

    "control_periods_ge_80pct",

    "control_percent_expected_ge_80pct",

]


print(
    final_summary[
        print_columns
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 34. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 120
)


print(
    "NOTEBOOK 13 COMPLETE"
)


print(
    "=" * 120
)


print(
    "\nOutput Excel:"
)


print(
    OUTPUT_EXCEL
)


print(
    "\nOutput CSV:"
)


print(
    OUTPUT_CSV
)


print(
    "\nExcel sheets:"
)


print(
    "README"
)


print(
    "Daily_S1"
)


print(
    "Daily_S2"
)


print(
    "Weekly_S1"
)


print(
    "Weekly_S2"
)


print(
    "Biweekly_S1"
)


print(
    "Biweekly_S2"
)


print(
    "\nPrimary threshold:"
)


print(
    ">= 80% valid pixels"
)


print(
    "\nNotebook completed successfully."
)

Packages loaded successfully.

All required source files found.

Treatment sites:
10

Controls per treatment:
matched_treatment_site_id
treatment_0001    5
treatment_0002    5
treatment_0003    5
treatment_0004    5
treatment_0005    5
treatment_0006    5
treatment_0007    5
treatment_0008    5
treatment_0009    5
treatment_0010    5
Name: site_id, dtype: int64

Simple Excel workbook saved:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/quality/image_quality_simple.xlsx

SIMPLE IMAGE QUALITY COMPARISON
 dataset    sensor period treatment_site  treatment_expected_periods  treatment_available_periods  treatment_periods_ge_80pct  treatment_percent_expected_ge_80pct  number_of_controls  control_expected_periods  control_available_periods  control_periods_ge_80pct  control_percent_expected_ge_80pct
   daily sentinel1 before treatment_0001                          11                           11                          11                                100.0    